In [22]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
import evaluate
from trl import SFTConfig, SFTTrainer
from peft import get_peft_model, LoraConfig, TaskType

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [24]:
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")['train']
dataset = dataset.filter(lambda x: x['output'] is not None and x['output'].strip() != "")
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 112165
})

In [25]:
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

In [26]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

In [27]:
def format_prompt(instruction, input_text, output_text=None, tokenizer=None):
    if input_text and input_text.strip():
        prompt = f"### Instruction: {instruction}\n### Input: {input_text}\n### Response:"
    else:
        prompt = f"### Instruction: {instruction}\n### Response:"
    if output_text is not None:
        if tokenizer is not None:
            prompt += f" {output_text}{tokenizer.eos_token}"
        else:
            prompt += f" {output_text}"
    return prompt

def tokenize_supervised(example, tokenizer, max_length=512):
    full_prompt = format_prompt(example["instruction"], example["input"], example["output"], tokenizer=tokenizer)
    prompt_only = format_prompt(example["instruction"], example["input"])

    prompt_only_ids = tokenizer(prompt_only, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = len(prompt_only_ids)

    tokenized = tokenizer(
        full_prompt,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    labels = tokenized["input_ids"].copy()
    if prompt_len >= max_length:
        return None
    labels[:prompt_len] = [-100] * prompt_len

    if all(l == -100 for l in labels):
        return None
    tokenized["labels"] = labels
    return tokenized

In [28]:
def tokenize_fn(example):
        return tokenize_supervised(example, tokenizer, max_length=1024)
tokenized_dataset = dataset.map(tokenize_fn, remove_columns=dataset.column_names)
tokenized_dataset = tokenized_dataset.filter(lambda x: x is not None and any(l != -100 for l in x['labels']))
print(f"Số sample còn lại: {len(tokenized_dataset)}")

Filter: 100%|██████████| 112142/112142 [01:13<00:00, 1525.97 examples/s]

Số sample còn lại: 112142


In [29]:
dataset_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split['train'].select(range(10000))
test_dataset = dataset_split['test'].select(range(1000))

In [30]:
model.eval()
prompt = format_prompt(dataset[0]['instruction'], dataset[0]['input'])

model.to(device)
print("Model device:", next(model.parameters()).device)

inputs = tokenizer(prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
for k, v in inputs.items():
    print(f"{k}: {v.device}")

with torch.no_grad():
    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
response = gen_text.split("### Response:")[-1].strip()
print(">>> Prompt:\n", prompt)
print(">>> Base model output:\n", response)
print(">>> Expected output:\n", dataset[0]['output'])

Model device: cuda:0
input_ids: cuda:0
attention_mask: cuda:0
>>> Prompt:
 ### Instruction: If you are a doctor, please answer the medical questions based on the patient's description.
### Input: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!
### Response:
>>> Base model output:
 Based on the patient's description, it seems that the patient is experiencing a severe case of vertigo. Vertigo is a condition characterized by a feeling of spinning or whirling, whi

In [31]:
sacrebleu = evaluate.load("sacrebleu")
results_base = sacrebleu.compute(predictions=[response], references=[dataset[0]['output']])
print("BLEU keys:", list(results_base.keys()))
print("BLEU score:", round(results_base["score"], 1))

BLEU keys: ['score', 'counts', 'totals', 'precisions', 'bp', 'sys_len', 'ref_len']
BLEU score: 2.7


In [32]:
lora_config = LoraConfig(
    r=16,  # Low-rank dimension
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Modules to apply LoRA
    lora_dropout=0.1,  # Dropout rate
    task_type=TaskType.CAUSAL_LM  # Task type should be causal language model
)

model = get_peft_model(model, lora_config)

In [33]:
collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

In [34]:
training_args = SFTConfig(
    output_dir="/tmp",
    num_train_epochs=1,
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    max_seq_length=1024,
    logging_steps=1000,
    logging_first_step=True,
    do_eval=True
)

trainer = SFTTrainer(
    model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=training_args,
    data_collator=collator,
)

Truncating eval dataset: 100%|██████████| 1000/1000 [00:00<00:00, 40762.95 examples/s]
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [35]:
import torch
torch.cuda.empty_cache()

In [36]:
trainer.train()

Step,Training Loss
1,2.727200
1000,2.370900
2000,2.274400
3000,2.274500
4000,2.250000
5000,2.257600


TrainOutput(global_step=5000, training_loss=2.285549045228958, metrics={'train_runtime': 1121.727, 'train_samples_per_second': 8.915, 'train_steps_per_second': 4.457, 'total_flos': 6.369885290496e+16, 'train_loss': 2.285549045228958})

In [37]:
trainer.save_model("/tmp")
tokenizer.save_pretrained("/tmp")

('/tmp/tokenizer_config.json',
 '/tmp/special_tokens_map.json',
 '/tmp/chat_template.jinja',
 '/tmp/tokenizer.model',
 '/tmp/added_tokens.json',
 '/tmp/tokenizer.json')

In [39]:
model.eval()
prompt = format_prompt(dataset[0]['instruction'], dataset[0]['input'])
inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    gen_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )
gen_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
response = gen_text.split("### Response:")[-1].strip()
print("Fine-tuned model output:\n", response)


Fine-tuned model output:
 Hi, I have gone through your query. You have mentioned that you are feeling the whole room is spinning when you are sitting down. This is a sign of vertigo. You should consult a neurologist for further evaluation. You should also consult a neurologist for further evaluation. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activities that can cause vertigo. You should avoid any activ